# 01 · The Post-Call Analysis (Summary) Agent

When a call ends, OutboundAI produces the **call walkthrough** you see in History: summary, sentiment, quality score,
what went well, what to improve, action items, and a **scheduled follow-up date**. This notebook rebuilds that
agent step by step.

**What you'll learn**

1. Where the agent sits in the call lifecycle
2. The prompt, the schema and `analyze_call`
3. Turning free-text ("next week") into a real follow-up date
4. The path for **unanswered** calls, where *no AI runs at all*
5. The whole pipeline end to end, then a **proposed upgrade**: lead-aware analysis that feeds new facts back into the lead

*Prerequisite:* skim `00_agent_foundations.ipynb` (provider, schema, retry, failure policy).

## 1. Where it sits in the real system

```
Twilio status webhook ──► save_and_cleanup(call_id, status)           (domain/calls/service.py)
                              │
          transcript exists?  ├─ yes ─► save transcript ─► asyncio.create_task(_run_analysis)
                              │                                   │
                              │                       analyze_call(transcript, contact_name)   ◄── THE AGENT
                              │                                   │
                              │                       parse_follow_up_date(analysis.follow_up_date_suggestion)
                              │                                   ▼
                              │                       call_logs.update(analysis, follow_up_date, follow_up_status="pending")
                              │
                              └─ no, status == "failed" ─► auto_schedule_unanswered_followup()  (rules only, no AI)
```

Key design points to notice:

* It runs as a **background task** after the call. The agent has no time pressure, so it can use a bigger model and
  a failure policy that *records* errors.
* The transcript is the **only** input today. (Section 8 shows why that is a limitation.)
* Every call ends with a stored `analysis` and `follow_up_status`, answered or not.

In [ ]:
!pip -q install pydantic openai

In [ ]:
# ---- Shared setup: identical in every notebook ---------------------------------------
# A trimmed copy of backend/app/ai/providers (base.py + factory.py). Agents only ever talk
# to an `LLMProvider`; they never know whether Groq, OpenAI or a fake is behind it.
import asyncio
import json
import os
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any, TypeVar

from pydantic import BaseModel, Field, ValidationError, field_validator

T = TypeVar("T", bound=BaseModel)


class LLMValidationError(RuntimeError):
    """complete_structured could not get valid JSON for the schema, even after one retry."""


@dataclass
class ProviderResponse:
    content: str | None


def _parse_json_content(content: str | None) -> Any:
    if not content:
        raise ValueError("empty response content")
    text = content.strip()
    if text.startswith("```"):  # models sometimes wrap JSON in a markdown fence
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:]
    return json.loads(text)


class LLMProvider(ABC):
    name: str

    @abstractmethod
    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None) -> ProviderResponse:
        """One chat-completion call."""

    async def complete_structured(self, messages, schema: type[T], *, model, temperature=0.3, max_tokens=1000) -> T:
        """Ask for JSON, validate it against `schema`, retry once with a fresh call on failure."""
        last_error: Exception | None = None
        for attempt in range(2):
            response = await self.complete(
                messages, model=model, temperature=temperature, max_tokens=max_tokens,
                response_format={"type": "json_object"},
            )
            try:
                return schema.model_validate(_parse_json_content(response.content))
            except (json.JSONDecodeError, ValidationError, ValueError, TypeError) as e:
                last_error = e
                print(f"  [retry] attempt {attempt + 1}/2 failed validation ({type(e).__name__})")
        raise LLMValidationError(f"{schema.__name__} validation failed after retry: {last_error}")


class OpenAICompatProvider(LLMProvider):
    """Real model. Groq exposes an OpenAI-compatible API, so one client class covers both here.
    (The repo has separate GroqProvider / OpenAIProvider classes; same idea.)"""

    def __init__(self, name: str, api_key: str, base_url: str | None = None):
        from openai import AsyncOpenAI

        self.name = name
        self._client = AsyncOpenAI(api_key=api_key, base_url=base_url)

    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None):
        kwargs = {"response_format": response_format} if response_format else {}
        start = time.monotonic()
        r = await self._client.chat.completions.create(
            model=model, messages=messages, temperature=temperature, max_tokens=max_tokens, **kwargs
        )
        print(f"  [LLM] provider={self.name} model={model} latency_ms={int((time.monotonic() - start) * 1000)}")
        return ProviderResponse(r.choices[0].message.content)


class FakeProvider(LLMProvider):
    """Offline stand-in: `responder(messages) -> str` returns canned/rule-based JSON text.
    This is how the repo's tests avoid real API calls (CODING_STANDARDS section 4)."""

    name = "fake"

    def __init__(self, responder):
        self.responder = responder
        self.calls: list[list[dict]] = []

    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None):
        self.calls.append(messages)
        return ProviderResponse(self.responder(messages))


GROQ_MODEL = "openai/gpt-oss-120b"  # settings.groq_model in the repo
GROQ_FAST_MODEL = "openai/gpt-oss-20b"  # settings.groq_fast_model (the small "gate" model)
OPENAI_MODEL = "gpt-4o"  # settings.openai_model


def _get_key(name: str) -> str | None:
    try:  # Colab: left sidebar -> key icon -> add a secret, enable notebook access
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)


def make_provider(fake_responder):
    """Use a real provider when GROQ_API_KEY / OPENAI_API_KEY is available, else fall back to
    FakeProvider(fake_responder) so the whole notebook still runs offline."""
    if key := _get_key("GROQ_API_KEY"):
        llm, model, fast = OpenAICompatProvider("groq", key, "https://api.groq.com/openai/v1"), GROQ_MODEL, GROQ_FAST_MODEL
    elif key := _get_key("OPENAI_API_KEY"):
        llm, model, fast = OpenAICompatProvider("openai", key), OPENAI_MODEL, OPENAI_MODEL
    else:
        llm, model, fast = FakeProvider(fake_responder), "fake-model", "fake-model"
    if llm.name == "fake":
        print("MODE: FAKE (offline). Outputs are canned/rule-based, NOT generated by a model.")
        print("      Add GROQ_API_KEY or OPENAI_API_KEY as a Colab secret to run for real.")
    else:
        print(f"MODE: LIVE via {llm.name} (model={model})")
    return llm, model, fast

## 2. Step 1: the input. A transcript in the app's own format

The backend builds the transcript as lines of `[Speaker]: text`. The caller (the BA) is always labelled `You`; the
contact is labelled by name. We will reuse this call throughout the notebook.

In [ ]:
CONTACT_NAME = "Rahul Mehta"

TRANSCRIPT = """\
[You]: Hi Rahul, this is the BA from SoftSuave. Is this a good time for a quick call?
[Rahul Mehta]: Yeah sure, I have about five minutes.
[You]: Great. I saw Northwind Logistics is expanding its warehouse network. We build warehouse management and tracking systems.
[Rahul Mehta]: Right, we're opening two new warehouses in Pune this year. Honestly our current setup is Excel and WhatsApp.
[You]: Have you looked at any WMS tools?
[Rahul Mehta]: We looked at a couple, too expensive. Budget is tight until Q3.
[You]: Understood. We built a custom tracking platform for a cold-chain distributor that cut dispatch errors by 30 percent.
[Rahul Mehta]: Interesting. Can you send a short case study? And call me next week, Tuesday maybe.
[You]: Absolutely, I'll email it today and call you Tuesday."""

print(TRANSCRIPT)

## 3. Step 2: the output contract (schema)

This is `CallAnalysis` from `app/ai/schemas/analysis.py`. Every field has a default, so a partially filled model reply
still validates. The UI then renders whatever is present.

In [ ]:
class CallAnalysis(BaseModel):
    summary: str = ""
    sentiment: str = "Neutral"
    quality_score: int = 0
    went_well: list[str] = Field(default_factory=list)
    to_improve: list[str] = Field(default_factory=list)
    action_items: list[str] = Field(default_factory=list)
    follow_up_needed: bool = False
    follow_up_reason: str = ""
    follow_up_date_suggestion: str = ""
    key_points: list[str] = Field(default_factory=list)


print(json.dumps(CallAnalysis.model_json_schema()["properties"], indent=1)[:700], "...")

## 4. Step 3: the prompt

Copied from `app/ai/prompts/analysis_prompts.py`. Three details are worth studying:

* It tells the model the **exact JSON shape**. Models follow shapes much more reliably than prose instructions.
* It forces **names over roles**: "refer to the client by name, never 'the prospect'". This keeps summaries readable
  in a CRM.
* `"Base everything on what was actually said"` is a cheap guard against invented content.

In [ ]:
ANALYSIS_PROMPT = """You are an expert call analyst. Analyze this completed call transcript between a BA (Business Agent) and {client_name}, a prospect/customer.

Return a JSON object with exactly these fields:
{{
  "summary": "2-3 sentence summary of what happened in the call - refer to the client by name ({client_name}), never as 'the prospect' or 'the client'",
  "sentiment": "Positive" or "Neutral" or "Negative",
  "quality_score": number from 1 to 10,
  "went_well": ["things the BA did well"],
  "to_improve": ["areas where the BA could improve"],
  "action_items": ["specific follow-up actions needed"],
  "follow_up_needed": true or false,
  "follow_up_reason": "why follow-up is needed (empty string if not needed)",
  "follow_up_date_suggestion": "suggested timeframe like 'in 2 days' or 'next week' (empty string if not needed)",
  "key_points": ["key topics and points discussed in the call"]
}}

Use "BA" (not "agent") whenever referring to the caller, and use {client_name}'s actual name (not "the prospect"/"the client") whenever referring to them. Be specific and actionable. Base everything on what was actually said in the transcript."""


def build_analysis_messages(transcript: str, contact_name: str = "") -> list[dict]:
    client_name = contact_name or "the client"
    return [
        {"role": "system", "content": ANALYSIS_PROMPT.format(client_name=client_name)},
        {"role": "user", "content": f"Call transcript:\n\n{transcript}\n\nAnalyze this call:"},
    ]


print(build_analysis_messages(TRANSCRIPT, CONTACT_NAME)[0]["content"][:420], "...")

## 5. Step 4: the agent itself

`analyze_call` is tiny, because the heavy lifting is in the provider, the schema and the prompt. Note the **failure
policy** from notebook 00: it never raises; it returns an `error` shape that gets stored on the call.

In FAKE mode the `responder` below stands in for the model. It returns a plausible analysis of *this* transcript so you
can see the downstream steps work. In LIVE mode it is ignored and the real model reads the transcript.

In [ ]:
def responder(messages):
    system = messages[0]["content"]
    if "lead_updates" in system:  # the proposed lead-aware prompt (section 8)
        return json.dumps(CANNED_V2)
    if "expert sales coach" in system:  # weekly summary (section 7)
        return json.dumps({
            "headline": "Warm week: most conversations ended with a concrete next step.",
            "wins": ["Strong opening permission check", "Case study offered at the right moment"],
            "areas_to_improve": ["Probe budget earlier", "Confirm decision-maker"],
            "top_priority_action": "Send promised case studies within 24 hours.",
            "outlook": "Positive",
        })
    return json.dumps(CANNED_V1)


CANNED_V1 = {
    "summary": "Rahul Mehta is opening two Pune warehouses and runs operations on Excel and WhatsApp. Rahul Mehta found WMS tools too expensive and has budget only after Q3, but asked the BA for a case study and a call next Tuesday.",
    "sentiment": "Positive",
    "quality_score": 8,
    "went_well": ["Asked permission before pitching", "Tied the cold-chain case study to Rahul Mehta's tracking problem"],
    "to_improve": ["Did not ask who else is involved in the decision", "Did not confirm a budget range"],
    "action_items": ["Email the cold-chain case study today", "Call Rahul Mehta on Tuesday"],
    "follow_up_needed": True,
    "follow_up_reason": "Rahul Mehta requested a case study and a call next week",
    "follow_up_date_suggestion": "next week",
    "key_points": ["Two new Pune warehouses", "Current process is Excel and WhatsApp", "Budget tight until Q3"],
}

llm, MODEL, FAST_MODEL = make_provider(responder)


async def analyze_call(transcript: str, contact_name: str = "") -> dict:
    """Generate post-call analysis from the complete transcript."""
    messages = build_analysis_messages(transcript, contact_name)
    try:
        result = await llm.complete_structured(messages, CallAnalysis, model=MODEL, temperature=0.3, max_tokens=1000)
        return result.model_dump()
    except Exception as e:
        print("[ANALYSIS] Failed:", e)
        return {"error": str(e), "summary": "Analysis failed."}


analysis = await analyze_call(TRANSCRIPT, CONTACT_NAME)
print(json.dumps(analysis, indent=2))

Read the output the way the History page does: summary → sentiment/score → key points → went well / to improve →
action items → follow-up banner. Every box in the UI is one schema field.

## 6. Step 5: from words to a date: `parse_follow_up_date`

The model says `"next week"`. The scheduler needs a `datetime`. The repo does this with **plain rules, not another
LLM call** (`domain/calls/follow_up.py`). Deterministic, free, instant, and easy to unit-test.

Copied faithfully below, then tested against realistic model phrasings.

In [ ]:
import re
from datetime import datetime, timedelta, timezone


def utcnow() -> datetime:
    """Naive UTC, the same value the repo gets from datetime.utcnow() (deprecated in newer Python)."""
    return datetime.now(timezone.utc).replace(tzinfo=None)


def parse_follow_up_date(suggestion: str, now: datetime | None = None) -> datetime:
    """Parse AI follow-up date suggestion like 'in 2 days', 'next week' into a real datetime."""
    now = now or utcnow()
    s = (suggestion or "").lower().strip()

    if not s:
        return now + timedelta(days=3)
    if "today" in s:
        return now
    if "tomorrow" in s:
        return now + timedelta(days=1)
    m = re.search(r"in\s+(\d+)\s+day", s)
    if m:
        return now + timedelta(days=int(m.group(1)))
    m = re.search(r"in\s+(\d+)\s+week", s)
    if m:
        return now + timedelta(weeks=int(m.group(1)))
    if "next week" in s:
        return now + timedelta(days=7)
    if "next month" in s:
        return now + timedelta(days=30)
    m = re.search(r"(\d+)\s+day", s)
    if m:
        return now + timedelta(days=int(m.group(1)))
    return now + timedelta(days=3)


FIXED_NOW = datetime(2026, 10, 22, 9, 0)  # a Thursday, so results are reproducible
phrases = ["", "today", "tomorrow", "in 2 days", "in 3 weeks", "next week", "next month", "5 days", "Tuesday afternoon", "sometime soon"]
print(f"{'model said':<20} -> follow-up date")
for p in phrases:
    print(f"{p!r:<20} -> {parse_follow_up_date(p, FIXED_NOW):%a %d %b %Y}")

Look at the last two rows: `"Tuesday afternoon"` and `"sometime soon"` both silently fall back to **+3 days**. In our
sample call Rahul said *Tuesday* and the BA agreed to call Tuesday, yet the rules cannot read weekdays. Worse, the
fallback landed on a **Sunday** (see the output above): the weekend nudge in Step 6 only applies to unanswered calls,
not to this path.

> **Discussion:** This is a real limitation of the rule-based design. You trade accuracy on unusual phrasing for
> speed, cost and testability. Options: (a) add weekday parsing to the rules, (b) ask the model for an ISO date
> directly, (c) both: ISO date first, rules as fallback. See Exercise 2.

## 7. Step 6: the path with **no AI**, unanswered calls

If nobody picks up (`no-answer`, `busy`, `failed`) there is no transcript, so nothing to analyze. Rather than let the
lead vanish, the backend fills the *same shape* by hand and schedules a follow-up two days out, **nudged off the
weekend to Monday**.

In [ ]:
def suggest_reschedule(now: datetime) -> tuple[datetime, str]:
    target = now + timedelta(days=2)
    if target.weekday() >= 5:  # Saturday=5, Sunday=6
        target += timedelta(days=7 - target.weekday())
        label = f"next Monday ({target.strftime('%b %d')})"
    else:
        label = f"in 2 days ({target.strftime('%b %d')})"
    return target, label


def unanswered_analysis(contact_name: str, now: datetime) -> tuple[dict, datetime]:
    target_date, label = suggest_reschedule(now)
    analysis = {
        "summary": f"{contact_name} did not answer the call. Follow-up suggested {label}.",
        "sentiment": "Neutral",
        "quality_score": None,
        "went_well": [],
        "to_improve": [],
        "action_items": [],
        "follow_up_needed": True,
        "follow_up_reason": f"{contact_name} did not answer the call",
        "follow_up_date_suggestion": label,
        "key_points": [],
    }
    return analysis, target_date


for day in [datetime(2026, 10, 20), datetime(2026, 10, 22), datetime(2026, 10, 23)]:
    target, label = suggest_reschedule(day)
    print(f"called on {day:%a %d %b} -> target {target:%a %d %b}  | {label}")

Called on Thursday → +2 days is Saturday → moved to Monday. Called Friday → Sunday → Monday. Only the Tuesday call
keeps its natural +2.

This is a good example of **"use an LLM only where rules cannot do the job"**. The same principle drives the lead
research agent, where only 3 of 13 nodes call a model.

## 8. Step 7: put it together, a faithful `save_and_cleanup`

This mirrors `domain/calls/service.py`. The repository is a plain dict here, standing in for the `call_logs`
MongoDB collection.

In [ ]:
call_logs: dict[str, dict] = {}  # stand-in for the call_logs collection


async def run_analysis(call_id: str, transcript: str, contact_name: str, now: datetime | None = None):
    analysis = await analyze_call(transcript, contact_name)
    update = {"analysis": analysis}
    if analysis.get("follow_up_needed"):
        update["follow_up_date"] = parse_follow_up_date(analysis.get("follow_up_date_suggestion", ""), now)
        update["follow_up_status"] = "pending"
    call_logs[call_id].update(update)


async def save_and_cleanup(call_id: str, status: str, transcript: str, contact_name: str, now: datetime):
    if transcript:
        call_logs[call_id]["transcript"] = transcript
        await run_analysis(call_id, transcript, contact_name, now)  # the repo runs this as a background task
    elif status == "failed":
        analysis, target = unanswered_analysis(contact_name, now)
        call_logs[call_id].update({"analysis": analysis, "follow_up_date": target, "follow_up_status": "pending"})


call_logs["c1"] = {"contact_name": CONTACT_NAME, "status": "completed"}
call_logs["c2"] = {"contact_name": "Anita Rao", "status": "failed"}

await save_and_cleanup("c1", "completed", TRANSCRIPT, CONTACT_NAME, FIXED_NOW)
await save_and_cleanup("c2", "failed", "", "Anita Rao", FIXED_NOW)

for cid, row in call_logs.items():
    a = row["analysis"]
    print(f"{cid}: {row['contact_name']:<12} sentiment={a['sentiment']:<8} follow-up {row['follow_up_date']:%a %d %b} ({row['follow_up_status']})")
    print("    ", a["summary"][:110], "...")

Two very different calls, one uniform record shape. That uniformity is why the dashboard's analytics queries
(`analysis.sentiment`, `follow_up_status`) work across all calls.

### The weekly roll-up (same agent file)

`generate_weekly_summary` shows a second use of the same pattern: build a text *digest* from many analyses, then ask
for a coaching summary. Different prompt, different schema, same machinery.

In [ ]:
class WeeklySummary(BaseModel):
    headline: str = ""
    wins: list[str] = Field(default_factory=list)
    areas_to_improve: list[str] = Field(default_factory=list)
    top_priority_action: str = ""
    outlook: str = "Neutral"


WEEKLY_SUMMARY_PROMPT = """You are an expert sales coach. Below is a digest of {call_count} outbound calls made this week.

{digest}

Return a JSON object with:
{{
  "headline": "one punchy sentence summarising the week",
  "wins": ["up to 3 things that went well across all calls"],
  "areas_to_improve": ["up to 3 coaching points"],
  "top_priority_action": "the single most important action for next week",
  "outlook": "Positive" or "Neutral" or "Needs Attention"
}}"""

lines = []
for row in call_logs.values():
    a = row["analysis"]
    lines.append(
        f"- {row['contact_name']} | sentiment:{a['sentiment']} | score:{a['quality_score']} "
        f"| follow_up:{a['follow_up_needed']} | actions: {'; '.join(a['action_items'])}"
    )
digest = "\n".join(lines)
print(digest, "\n")

weekly = await llm.complete_structured(
    [{"role": "user", "content": WEEKLY_SUMMARY_PROMPT.format(digest=digest, call_count=len(lines))}],
    WeeklySummary, model=MODEL, temperature=0.4, max_tokens=600,
)
print(weekly.model_dump_json(indent=2))

## 9. A limitation, then a proposed upgrade (not in the repo yet)

Today `analyze_call(transcript, contact_name)` sees **only the transcript**. Consider what it cannot do:

* It cannot say *"the pain point we predicted from research was confirmed"* because it has never seen the research.
* It cannot tell *new facts* ("two new warehouses", "budget after Q3") apart from noise, so nothing learned on the call
  flows back to the lead record. The summary just sits in History.
* It does not know the previous call, so repeat conversations are summarised in isolation.

The upgrade has three parts, all small:

1. **Pass lead context** into the prompt (company, research hypotheses, matched case studies, last call summary).
2. **Extend the schema** with `hypothesis_check` and `lead_updates`.
3. **Write back safely**: fill blank lead fields; turn conflicts into *suggestions* a human accepts.

This is the bridge to the lead research agent (notebook 03): research produces hypotheses before the call; the call
confirms or contradicts them; the result updates the brief.

In [ ]:
class LeadContext(BaseModel):
    company: str = ""
    domain: str = ""
    hypotheses: list[str] = Field(default_factory=list)  # pain points predicted by pre-call research
    matched_projects: list[str] = Field(default_factory=list)
    previous_summary: str = ""


class HypothesisCheck(BaseModel):
    hypothesis: str
    verdict: str  # "confirmed" | "contradicted" | "not_discussed"
    evidence: str = ""  # short quote from the transcript


class LeadUpdate(BaseModel):
    field: str  # e.g. "role", "budget_timeline", "tech_in_use", "company_size"
    value: str
    evidence: str = ""  # quote from the transcript proving the contact said it


class CallAnalysisV2(CallAnalysis):
    hypothesis_check: list[HypothesisCheck] = Field(default_factory=list)
    lead_updates: list[LeadUpdate] = Field(default_factory=list)


V2_EXTRA = """

PRE-CALL CONTEXT about {client_name} (from our research - may be wrong):
- Company: {company} ({domain})
- Hypotheses to check: {hypotheses}
- Our relevant past projects: {projects}
- Previous call summary: {previous}

Also return these two extra fields in the same JSON object:
  "hypothesis_check": [{{"hypothesis": "...", "verdict": "confirmed" or "contradicted" or "not_discussed", "evidence": "short quote from the transcript"}}],
  "lead_updates": [{{"field": "role|budget_timeline|tech_in_use|company_size|competitor|decision_maker", "value": "...", "evidence": "short quote from the transcript"}}]

Rules for lead_updates: include ONLY facts the contact stated in this call, each with a verbatim quote. Never infer. Return an empty list if none."""


def build_analysis_messages_v2(transcript: str, contact_name: str, ctx: LeadContext) -> list[dict]:
    base = build_analysis_messages(transcript, contact_name)
    extra = V2_EXTRA.format(
        client_name=contact_name or "the client",
        company=ctx.company or "unknown",
        domain=ctx.domain or "unknown",
        hypotheses="; ".join(ctx.hypotheses) or "none",
        projects="; ".join(ctx.matched_projects) or "none",
        previous=ctx.previous_summary or "none",
    )
    base[0]["content"] += extra
    return base


CANNED_V2 = {
    **CANNED_V1,
    "hypothesis_check": [
        {"hypothesis": "Dispatch and inventory are coordinated manually", "verdict": "confirmed", "evidence": "current setup is Excel and WhatsApp"},
        {"hypothesis": "No dedicated IT leader", "verdict": "not_discussed", "evidence": ""},
        {"hypothesis": "Cost is not a concern", "verdict": "contradicted", "evidence": "too expensive. Budget is tight until Q3"},
    ],
    "lead_updates": [
        {"field": "budget_timeline", "value": "Budget available after Q3", "evidence": "Budget is tight until Q3"},
        {"field": "tech_in_use", "value": "Excel and WhatsApp", "evidence": "current setup is Excel and WhatsApp"},
    ],
}

ctx = LeadContext(
    company="Northwind Logistics",
    domain="northwind-logistics.example",
    hypotheses=["Dispatch and inventory are coordinated manually", "No dedicated IT leader", "Cost is not a concern"],
    matched_projects=["ColdChain Track (logistics)"],
    previous_summary="",
)

v2 = await llm.complete_structured(
    build_analysis_messages_v2(TRANSCRIPT, CONTACT_NAME, ctx), CallAnalysisV2, model=MODEL, temperature=0.3, max_tokens=1500
)
print("hypothesis_check:")
for h in v2.hypothesis_check:
    print(f"  [{h.verdict:<13}] {h.hypothesis}   <- {h.evidence!r}")
print("lead_updates:")
for u in v2.lead_updates:
    print(f"  {u.field:<16} = {u.value}   <- {u.evidence!r}")

### Make the evidence checkable

Asking for a verbatim `evidence` quote is not decoration. It lets code **verify the model did not invent a fact**: if
the quote is not in the transcript, drop the update. This is the same "grounding" idea used in the lead research agent.

In [ ]:
def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", s.lower()).strip()


def grounded_updates(updates: list[LeadUpdate], transcript: str) -> tuple[list[LeadUpdate], list[LeadUpdate]]:
    text = _norm(transcript)
    kept, dropped = [], []
    for u in updates:
        (kept if u.evidence and _norm(u.evidence) in text else dropped).append(u)
    return kept, dropped


# Pretend the model hallucinated one extra fact:
suspect = v2.lead_updates + [LeadUpdate(field="company_size", value="500 employees", evidence="we have 500 people")]
kept, dropped = grounded_updates(suspect, TRANSCRIPT)
print("kept   :", [u.field for u in kept])
print("dropped:", [(u.field, u.value) for u in dropped], "<- quote not found in transcript")

### Write back without clobbering a human's edits

Policy (mirrors `_upsert_contact`'s "only fill blanks" rule in `routers/clients.py`):

* Field is **blank** on the lead → fill it automatically, with provenance.
* Field **already has a different value** → do not overwrite; create a **pending suggestion** for the user.

In [ ]:
def apply_lead_updates(lead: dict, updates: list[LeadUpdate], call_id: str) -> dict:
    applied, suggestions = [], []
    for u in updates:
        current = lead.get(u.field, "")
        record = {"field": u.field, "value": u.value, "evidence": u.evidence, "source": f"call:{call_id}"}
        if not current:
            lead[u.field] = u.value
            applied.append(record)
        elif _norm(current) != _norm(u.value):
            suggestions.append({**record, "current": current, "status": "pending"})
    return {"applied": applied, "suggestions": suggestions}


lead = {"name": CONTACT_NAME, "company": "Northwind Logistics", "tech_in_use": "Tally + paper", "budget_timeline": ""}
outcome = apply_lead_updates(lead, kept, call_id="c1")

print("auto-applied :", [(a["field"], a["value"]) for a in outcome["applied"]])
print("needs review :", [(s["field"], f'{s["current"]!r} -> {s["value"]!r}') for s in outcome["suggestions"]])
print("lead now     :", lead)

`budget_timeline` was blank → filled. `tech_in_use` already said "Tally + paper" → left alone and queued as a suggestion.

**Trade-off to remember:** auto-filling blanks is fast and low-risk; auto-overwriting is fast and high-risk (a
mis-heard number replaces good data). The two-tier policy keeps most of the speed and removes most of the risk.

## 10. Known issues in the current implementation (good review practice)

Reading code critically is part of learning it. In the repo today:

| Issue | Where | Why it matters |
|---|---|---|
| Analysis is triggered from **two places** with duplicated logic | `domain/calls/service.py::_run_analysis` and `routers/calls.py::_trigger_analysis` | Behaviour can drift; the second is a side effect of a `GET /calls/logs` request |
| Retry counters live in module-level dicts | `routers/calls.py` (`_analysis_attempts`) | Reset on restart; do not work across multiple server processes |
| Background task started with a bare `asyncio.create_task` | `service.py` | If the process dies mid-analysis, that call has no analysis until someone opens History |
| Follow-up parsing cannot read weekdays | `follow_up.py` | The sample call says "Tuesday" and we schedule +3 days |

None are bugs you must fix today, but each is a place where a small change (one idempotent `ensure_analysis`
service function) removes a class of problems.

## 11. Exercises

1. **Run it live.** Add a Groq or OpenAI key and re-run Step 4. Compare the real analysis to the canned one. Did the
   model use the client's name as instructed?
2. **Fix the weekday gap.** Extend `parse_follow_up_date` so `"Tuesday"`, `"next Tuesday"` and `"Tuesday afternoon"`
   resolve to the next Tuesday. Write 5 assertions first.
3. **Stress the schema.** Make `FakeProvider` return `{"quality_score": "eight"}`. What happens? Should
   `CallAnalysis` tolerate it? (Review the "tolerant schema" trade-off in notebook 00.)
4. **Extend lead updates.** Add a `decision_maker` rule: if a quote contains "my boss", "my manager" or "board", set
   `decision_maker="someone else"`. Where would that rule belong: the prompt, the schema, or `apply_lead_updates`?
5. **Idempotency.** Write `ensure_analysis(call_id)` that does nothing if an analysis without an `error` already
   exists, and is safe to call twice concurrently. (Hint: one `asyncio.Lock` per call id.)

## Where this lives in the repo

| Concept | File |
|---|---|
| Agent (`analyze_call`, weekly summary) | `backend/app/ai/agents/analysis_agent.py` |
| Prompts / schema | `backend/app/ai/prompts/analysis_prompts.py`, `backend/app/ai/schemas/analysis.py` |
| Lifecycle (`save_and_cleanup`, `_run_analysis`) | `backend/app/domain/calls/service.py` |
| Follow-up rules | `backend/app/domain/calls/follow_up.py` |
| UI rendering | `frontend/src/features/history/components/CallHistory.jsx` |

**Next:** `02_live_suggestion_agent.ipynb`